In [ ]:

from pathlib import Path
ROOT = Path(r"../data/MP_640/")  
VIDEO_EXTS = {".mp4", ".mov", ".mkv", ".avi"}
FACE_TASK_MODEL = Path(r"model/face_landmarker.task")
TARGET_FPS = 10  
BACKBONE_SUFFIX = "_mediapipe_backbone.csv"
FACE_SELECTION_STRATEGY = "max_presence"  

In [ ]:
BLENDSHAPE_NAMES = [
    "browDownLeft","browDownRight","browInnerUp","browOuterUpLeft","browOuterUpRight",
    "cheekPuff","cheekSquintLeft","cheekSquintRight",
    "eyeBlinkLeft","eyeBlinkRight","eyeLookDownLeft","eyeLookDownRight",
    "eyeLookInLeft","eyeLookInRight","eyeLookOutLeft","eyeLookOutRight",
    "eyeLookUpLeft","eyeLookUpRight","eyeSquintLeft","eyeSquintRight","eyeWideLeft","eyeWideRight",
    "jawForward","jawLeft","jawOpen","jawRight",
    "mouthClose","mouthFrownLeft","mouthFrownRight","mouthFunnel",
    "mouthLeft","mouthLowerDownLeft","mouthLowerDownRight","mouthPressLeft","mouthPressRight",
    "mouthPucker","mouthRight","mouthRollLower","mouthRollUpper","mouthShrugLower","mouthShrugUpper",
    "mouthSmileLeft","mouthSmileRight","mouthStretchLeft","mouthStretchRight",
    "mouthUpperUpLeft","mouthUpperUpRight","noseSneerLeft","noseSneerRight","tongueOut","cheekSquint"
]

In [ ]:
POSE_KEEP = {
    "leftShoulder": 11,
    "rightShoulder": 12,
    "leftElbow": 13,
    "rightElbow": 14,
    "leftWrist": 15,
    "rightWrist": 16,
}
POSE_COORDS = ["x","y","z","visibility"]

def pose_cols():
    cols = []
    for name in POSE_KEEP:
        for c in POSE_COORDS:
            cols.append(f"{name}_{c}")
    return cols


In [ ]:
def find_videos(root: Path):
    vids = []
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in VIDEO_EXTS:
            vids.append(p)
    return vids

videos = find_videos(ROOT)
print(f"Found {len(videos)} videos")
for v in videos[:5]:
    print("•", v)


In [ ]:
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

import mediapipe as mp
mp_pose = mp.solutions.pose

assert hasattr(mp, "Image") and hasattr(mp, "ImageFormat"), "mp.Image or mp.ImageFormat missing; check mediapipe version."

def create_face_landmarker(model_path: Path):
    path = Path(model_path).expanduser().resolve()
    if not path.exists():
        raise FileNotFoundError(
            f"FaceLandmarker model not found: {path}\n"
        )
    with open(path, "rb") as f:
        model_bytes = f.read()

    base_opt = mp_python.BaseOptions(
        model_asset_buffer=model_bytes  
        
    )
    opt = mp_vision.FaceLandmarkerOptions(
        base_options=base_opt,
        output_facial_transformation_matrixes=False,
        num_faces=2,
        running_mode=mp_vision.RunningMode.IMAGE,
        output_face_blendshapes=True,
    )
    return mp_vision.FaceLandmarker.create_from_options(opt)


def select_face_index(result):
    bs_list = getattr(result, "face_blendshapes", None)
    if not bs_list:
        return None
    if FACE_SELECTION_STRATEGY == "first":
        return 0

    def sum_scores(bs_entry):
        cats = getattr(bs_entry, "categories", bs_entry)  
        return sum(getattr(c, "score", 0.0) for c in (cats or []))

    totals = [sum_scores(e) for e in bs_list]
    return int(np.argmax(totals)) if totals else None


def extract_blendshapes(face_landmarker, frame_bgr):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)  
    result = face_landmarker.detect(image)
    idx = select_face_index(result)
    if idx is None:
        return None

    bs_entry = result.face_blendshapes[idx]
    categories = getattr(bs_entry, "categories", bs_entry) or []  

    name_to_score = {}
    for cat in categories:
        name = getattr(cat, "category_name", None) or getattr(cat, "display_name", None) or ""
        if name:
            name_to_score[name] = float(getattr(cat, "score", 0.0))

    vec = np.zeros(len(BLENDSHAPE_NAMES), dtype=np.float32)
    for i, nm in enumerate(BLENDSHAPE_NAMES):
        vec[i] = name_to_score.get(nm, np.nan)  
    return vec


def extract_pose_subset(pose_obj, frame_bgr):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    res = pose_obj.process(rgb)
    out = {f"{k}_{c}": np.nan for k in POSE_KEEP for c in POSE_COORDS}
    if res.pose_landmarks is None:
        return out
    lm = res.pose_landmarks.landmark
    for name, idx in POSE_KEEP.items():
        pt = lm[idx]
        out[f"{name}_x"] = float(pt.x)
        out[f"{name}_y"] = float(pt.y)
        out[f"{name}_z"] = float(pt.z)
        out[f"{name}_visibility"] = float(pt.visibility)
    return out


def iter_video_frames(cap, target_fps=None):
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    if target_fps is None or target_fps <= 0:
        frame_idx = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            ts = frame_idx / max(fps, 1e-6)
            yield frame_idx, ts, frame
            frame_idx += 1
    else:
        step = int(round(fps / target_fps))
        if step <= 0:
            step = 1
        frame_idx = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if frame_idx % step == 0:
                ts = frame_idx / max(fps, 1e-6)
                yield frame_idx, ts, frame
            frame_idx += 1


In [ ]:
def process_video(video_path: Path):
    out_csv = video_path.with_name(video_path.stem + BACKBONE_SUFFIX)
    
    
    if out_csv.exists():
        print(f"[SKIP] {out_csv} already exists")
        return
    
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"[ERROR] Cannot open video: {video_path}")
        return

    try:
        face_lm = create_face_landmarker(FACE_TASK_MODEL)
    except Exception as e:
        print(f"[ERROR] FaceLandmarker init failed: {e}")
        cap.release()
        return

    with mp_pose.Pose(static_image_mode=False, model_complexity=1, enable_segmentation=False) as pose_obj:
        rows = []
        for frame_idx, ts, frame in tqdm(iter_video_frames(cap, TARGET_FPS), desc=video_path.name):
            
            bs_vec = extract_blendshapes(face_lm, frame)

            
            pose_dict = extract_pose_subset(pose_obj, frame)

            row = {
                "frame_idx": frame_idx,
                "timestamp_sec": ts,
                "video_path": str(video_path),
            }
            
            if bs_vec is not None:
                for i, nm in enumerate(BLENDSHAPE_NAMES):
                    row[nm] = float(bs_vec[i])
            else:
                for nm in BLENDSHAPE_NAMES:
                    row[nm] = np.nan

            
            row.update(pose_dict)
            rows.append(row)

    cap.release()
    if not rows:
        print(f"No frames processed for {video_path}")
        return

    df = pd.DataFrame(rows)
    
    cols = ["frame_idx","timestamp_sec","video_path"] + BLENDSHAPE_NAMES + pose_cols()
    for c in cols:
        if c not in df.columns:
            df[c] = np.nan
    df = df[cols]
    df.to_csv(out_csv, index=False)
    print(f"Saved backbone: {out_csv}")


for v in videos:
    process_video(v)
